# 00 - Setup and Data Audit
**Addresses reviewer comments 1, 2, 4 (foundation).**  Nothing here trains a model.

1. Builds the **train / validation / test manifest** (80/10/10, fixed seed, saved as JSON). The validation split is used only for
   checkpoint selection; the test split is used once per run. (The submitted paper selected checkpoints, tuned hyper-parameters
   and reported results on the same 320 images.)
2. Audits the data: value ranges, the **definition of the reference residual** `r = y - D(x)`, and a contact sheet of random ground-truth
   images so the paper can state truthfully what the data look like.
3. **Estimates the degradation parameters (speckle looks, Gaussian sigma) from the training pairs** instead of hard-coding them.

Set `SEMICON_MODE` (`smoke` / `pilot` / `full`) and `SEMICON_DATA` before launching Jupyter - see README.

In [ ]:
import os, sys, json, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
import config as C
C.banner()
from utils.data import load_npy, downsample
from utils.degrade import fit_degradation
from utils import figures as FG

MODE=pilot  epochs=30  seeds=[0, 1]  device=cpu  out=d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_pilot


## 1. Split manifest

In [ ]:
splits, man = C.get_splits()
print({k: len(v) for k, v in splits.items()}, "| manifest sha1:", man["sha1"][:12])
assert not (set(man["train"]) & set(man["val"])) and not (set(man["train"]) & set(man["test"])) and not (set(man["val"]) & set(man["test"]))
print("OK: train/val/test are disjoint. Manifest saved to", os.path.join(C.OUT, "split_manifest.json"))
env = dict(torch=torch.__version__, cuda=torch.cuda.is_available(),
           gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
           mode=C.MODE, epochs=C.EPOCHS, seeds=C.SEEDS, batch_size=C.BATCH_SIZE, lr=C.LR, split_sha1=man["sha1"])
json.dump(env, open(os.path.join(C.TABLES, "environment.json"), "w"), indent=1); print(env)

{'train': 2560, 'val': 320, 'test': 320} | manifest sha1: 31d7258b8c36
OK: train/val/test are disjoint. Manifest saved to d:\RESEARCH PAPERS\Semi_Conductor_2\Semi conductor image paper\KowHow\results_pilot\split_manifest.json
{'torch': '2.13.0+cpu', 'cuda': False, 'gpu': None, 'mode': 'pilot', 'epochs': 30, 'seeds': [0, 1], 'batch_size': 8, 'lr': 0.0002, 'split_sha1': '31d7258b8c36e2fd1262b33776880d7ec6ed4ce2'}


## 2. Value ranges and the reference residual

In [ ]:
rng = np.random.RandomState(0)
sample = [splits["train"][i] for i in rng.choice(len(splits["train"]), min(200, len(splits["train"])), replace=False)]
rows = []
for g, n in sample:
    gt, ny = load_npy(g), load_npy(n); ds = downsample(gt, ny.shape[:2])
    rows.append(dict(gt_min=gt.min(), gt_max=gt.max(), noisy_min=ny.min(), noisy_max=ny.max(),
                     frac_noisy_below0=(ny < 0).mean(), frac_noisy_above1=(ny > 1).mean(),
                     res_mean=(ny - ds).mean(), res_std=(ny - ds).std()))
audit = pd.DataFrame(rows)
print(audit.describe().loc[["mean", "min", "max"]].round(4).to_string())
print("\nGT shape:", gt.shape, " noisy shape:", ny.shape, "-> scale factor", gt.shape[0] // ny.shape[0])
print("NOTE: degraded values leave [0,1]; metrics clamp predictions to [0,1] and use data_range=1 (as the paper states).")
audit.describe().round(5).to_csv(os.path.join(C.TABLES, "data_audit.csv"))

      gt_min  gt_max  noisy_min  noisy_max  frac_noisy_below0  frac_noisy_above1  res_mean  res_std
mean     0.0     1.0    -0.0056     1.3896             0.0033             0.0358   -0.0000   0.0832
min      0.0     1.0    -0.1268     0.7221             0.0000             0.0000   -0.0024   0.0114
max      0.0     1.0     0.1115     1.9384             0.2163             0.3748    0.0018   0.1636

GT shape: (256, 256)  noisy shape: (128, 128) -> scale factor 2
NOTE: degraded values leave [0,1]; metrics clamp predictions to [0,1] and use data_range=1 (as the paper states).


In [ ]:
# Is the reference downsampler D(.) the one the data provider used?  If not, 'residual' contains structured edge error.
# Diagnostic: residual std vs. integer misregistration of D(x) relative to y.  The minimum should be at (0,0).
import torch.nn.functional as F
from scipy import ndimage
def resid_std(shift, mode="area"):
    out = []
    for g, n in sample[:60]:
        gt, ny = load_npy(g), load_npy(n)
        if mode == "area": ds = downsample(gt, ny.shape[:2])
        elif mode == "bilinear": ds = F.interpolate(torch.from_numpy(gt)[None, None], size=ny.shape[:2], mode="bilinear", align_corners=False)[0, 0].numpy()
        else: ds = ndimage.zoom(gt, ny.shape[0] / gt.shape[0], order=1)      # what the 'validates the statistical claim' notebooks used
        ds = np.roll(ds, shift, axis=(0, 1)); out.append((ny - ds)[2:-2, 2:-2].std())
    return float(np.mean(out))
grid = pd.DataFrame({f"dx={dx}": [resid_std((dy, dx)) for dy in (-1, 0, 1)] for dx in (-1, 0, 1)}, index=[f"dy={d}" for d in (-1, 0, 1)])
print("residual std vs misregistration (area-average D):\n", grid.round(5).to_string())
modes = {m: resid_std((0, 0), m) for m in ("area", "bilinear", "zoom_order1")}
print("\nresidual std by choice of D:", {k: round(v, 5) for k, v in modes.items()})
best = np.unravel_index(grid.values.argmin(), grid.shape)
print("minimum at", grid.index[best[0]], grid.columns[best[1]], "->", "OK (aligned)" if best == (1, 1) else "WARNING: reference D is mis-registered/different; report it")
grid.to_csv(os.path.join(C.TABLES, "residual_definition_check.csv"))

residual std vs misregistration (area-average D):
          dx=-1     dx=0     dx=1
dy=-1  0.12072  0.10846  0.12249
dy=0   0.10959  0.08334  0.11022
dy=1   0.12201  0.10820  0.12035

residual std by choice of D: {'area': 0.08334, 'bilinear': 0.08334, 'zoom_order1': 0.07964}
minimum at dy=0 dx=0 -> OK (aligned)


## 3. What do the images look like?  (state it honestly in the paper)
The submitted figures from the KLA data show a bristled, specimen-like grayscale object, not an obvious wafer pattern. A semiconductor-manufacturing journal will ask. Look at the sheet and describe the data accurately (and, if it is not wafer imagery, say so and rely on the Carinthia-S SEM experiments for domain claims).

In [ ]:
idx = np.random.RandomState(1).choice(len(splits["train"]), min(16, len(splits["train"])), replace=False)
fig, ax = plt.subplots(4, 4, figsize=(8, 8))
names = []
for a, i in zip(ax.ravel(), idx):
    g, _ = splits["train"][i]; a.imshow(load_npy(g), cmap="gray", vmin=0, vmax=1); a.axis("off"); names.append(os.path.basename(g))
FG.save(fig, C.FIGS, "fig_data_contact_sheet", {"split": "train", "files": names, "selection": "random, seed=1"})
print("saved contact sheet:", names[:4], "...")

saved contact sheet: ['001772.npy', '003121.npy', '000570.npy', '002105.npy'] ...


## 4. Estimate the degradation parameters from TRAIN pairs only
Moment matching `E[r^2 | I] = a I^2 + b` on flat regions (a = 1/(4 L), b = sigma^2/4). `r2_signal_dependence` tells you whether the model form fits at all.

In [ ]:
fit = fit_degradation(splits["train"], n_images=min(400, len(splits["train"])))
json.dump(fit, open(os.path.join(C.TABLES, "degradation_fit.json"), "w"), indent=1)
print(json.dumps(fit, indent=1))
print(f"\nUse looks={fit['looks']:.2f}, sigma={fit['sigma']:.4f} for ALL synthetic degradations (robustness / OOD / downstream).")
if fit["r2_signal_dependence"] < 0.8:
    print("WARNING: weak signal-dependence fit - the multiplicative-speckle + Gaussian model may not describe this data; say so in the paper.")

{
 "looks": 9.259709358215332,
 "sigma": 0.04252737760543823,
 "a": 0.02699868753552437,
 "b": 0.00045214444980956614,
 "r2_signal_dependence": 0.9977861642837524,
 "mean_residual": 8.677723235450685e-05,
 "n_pixels": 2621600,
 "note": "moment-matching estimate; check r2_signal_dependence before trusting looks/sigma"
}

Use looks=9.26, sigma=0.0425 for ALL synthetic degradations (robustness / OOD / downstream).
